# Dataset Info

Overview of the YC dataset: what we have, how fresh it is, and what it looks like.

In [1]:
import pandas as pd
import sqlite3
import os
from datetime import datetime

DB_PATH = "../data/yc_collaboration.db"
CSV_PATH = "../data/yc_collaboration_companies.csv"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM companies", conn)
conn.close()

print("Dataset loaded.")

Dataset loaded.


---
## 1. Data Source & Freshness

In [2]:
db_modified = datetime.fromtimestamp(os.path.getmtime(DB_PATH)).strftime("%Y-%m-%d %H:%M")
csv_modified = datetime.fromtimestamp(os.path.getmtime(CSV_PATH)).strftime("%Y-%m-%d %H:%M")
db_size = os.path.getsize(DB_PATH) / (1024 * 1024)
csv_size = os.path.getsize(CSV_PATH) / (1024 * 1024)

info = {
    "Source": "yc-oss.github.io/api (community-maintained, from YC Algolia index)",
    "Source last updated": "2026-02-08",
    "Our scrape date": db_modified,
    "SQLite file": f"{DB_PATH} ({db_size:.2f} MB)",
    "CSV file": f"{CSV_PATH} ({csv_size:.2f} MB)",
    "Total companies": len(df),
}

for k, v in info.items():
    print(f"{k:<25} {v}")

Source                    yc-oss.github.io/api (community-maintained, from YC Algolia index)
Source last updated       2026-02-08
Our scrape date           2026-03-11 12:22
SQLite file               ../data/yc_collaboration.db (5.52 MB)
CSV file                  ../data/yc_collaboration_companies.csv (4.28 MB)
Total companies           5690


---
## 2. Schema & Columns

In [3]:
col_info = pd.DataFrame({
    "Column": df.columns,
    "Type": df.dtypes.values,
    "Non-Null": df.notnull().sum().values,
    "Null": df.isnull().sum().values,
    "Fill %": (df.notnull().sum().values / len(df) * 100).round(1),
    "Unique": df.nunique().values,
    "Sample": [df[col].dropna().iloc[0] if df[col].notnull().any() else "" for col in df.columns],
})

# Truncate sample values for display
col_info["Sample"] = col_info["Sample"].astype(str).str[:60]

print(f"Columns: {len(df.columns)}")
print(f"Rows: {len(df):,}")
print()
col_info

Columns: 17
Rows: 5,690



,Column,Type,Non-Null,Null,Fill %,Unique,Sample
0,id,int64,5690,0,100.0,5690,5
1,name,str,5690,0,100.0,5606,CircuitHub
2,one_liner,str,5690,0,100.0,5531,On-Demand Electronics Manufacturing
3,long_description,str,5660,30,99.5,5317,CircuitHub offers on-demand electronics manufa...
4,website,str,5689,1,100.0,5653,https://circuithub.com
5,batch,str,5690,0,100.0,48,Winter 2012
6,status,str,5690,0,100.0,4,Active
7,team_size,float64,5590,100,98.2,199,58.0
8,is_hiring,int64,5690,0,100.0,2,1
9,location,str,5690,0,100.0,708,"London, England, United Kingdom"


---
## 3. Status Breakdown

In [4]:
status = df["status"].value_counts().reset_index()
status.columns = ["Status", "Count"]
status["Percentage"] = (status["Count"] / status["Count"].sum() * 100).round(1).astype(str) + "%"
status

,Status,Count,Percentage
0,Active,3930,69.1%
1,Inactive,1005,17.7%
2,Acquired,732,12.9%
3,Public,23,0.4%


---
## 4. Batch Range & Distribution

In [5]:
df["batch_year"] = df["batch"].str.extract(r"(\d{4})").astype(float)

batches = df["batch"].value_counts().sort_index()
years = df["batch_year"].dropna()

print(f"{'Oldest batch':<25} {batches.index[0]} ({int(batches.iloc[0])} companies)")
print(f"{'Newest batch':<25} {batches.index[-1]} ({int(batches.iloc[-1])} companies)")
print(f"{'Total batches':<25} {len(batches)}")
print(f"{'Year range':<25} {int(years.min())} - {int(years.max())} ({int(years.max() - years.min())} years)")
print()

# Top 5 largest batches
print("=== Largest Batches ===")
for batch, count in batches.sort_values(ascending=False).head(5).items():
    print(f"  {batch:<20} {count} companies")

Oldest batch              Fall 2024 (93 companies)
Newest batch              Winter 2026 (132 companies)
Total batches             48
Year range                2005 - 2026 (21 years)

=== Largest Batches ===
  Winter 2022          399 companies
  Summer 2021          391 companies
  Winter 2021          336 companies
  Winter 2023          274 companies
  Winter 2024          251 companies


In [6]:
# Companies per year
yearly = df.groupby("batch_year").size()

print("=== Companies per Year ===")
for year, count in yearly.items():
    bar = "|" * int(count / 10)
    print(f"  {int(year)}  {count:>4}  {bar}")

=== Companies per Year ===
  2005     9  
  2006    18  |
  2007    32  |||
  2008    43  ||||
  2009    42  ||||
  2010    63  ||||||
  2011   104  ||||||||||
  2012   149  ||||||||||||||
  2013    98  |||||||||
  2014   152  |||||||||||||||
  2015   216  |||||||||||||||||||||
  2016   224  ||||||||||||||||||||||
  2017   241  ||||||||||||||||||||||||
  2018   277  |||||||||||||||||||||||||||
  2019   371  |||||||||||||||||||||||||||||||||||||
  2020   437  |||||||||||||||||||||||||||||||||||||||||||
  2021   727  ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  2022   633  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  2023   495  |||||||||||||||||||||||||||||||||||||||||||||||||
  2024   593  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  2025   631  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  2026   134  |||||||||||||


---
## 5. Industry Breakdown

In [7]:
industries = df["industry"].value_counts().reset_index()
industries.columns = ["Industry", "Count"]
industries["Percentage"] = (industries["Count"] / industries["Count"].sum() * 100).round(1).astype(str) + "%"
industries

,Industry,Count,Percentage
0,B2B,2876,50.5%
1,Consumer,866,15.2%
2,Healthcare,656,11.5%
3,Fintech,607,10.7%
4,Industrials,349,6.1%
5,Real Estate and Construction,153,2.7%
6,Education,125,2.2%
7,Government,40,0.7%
8,Unspecified,18,0.3%


---
## 6. Stage Breakdown

In [8]:
stage = df["stage"].value_counts().reset_index()
stage.columns = ["Stage", "Count"]
stage["Percentage"] = (stage["Count"] / stage["Count"].sum() * 100).round(1).astype(str) + "%"
stage

,Stage,Count,Percentage
0,Early,4652,81.8%
1,Growth,1038,18.2%


---
## 7. Hiring Overview

In [9]:
total = len(df)
hiring = int(df["is_hiring"].sum())
active_hiring = int(df[(df["is_hiring"] == 1) & (df["status"] == "Active")].shape[0])

print(f"{'Total companies':<30} {total:,}")
print(f"{'Currently hiring':<30} {hiring:,} ({hiring/total*100:.1f}%)")
print(f"{'Active & hiring':<30} {active_hiring:,} ({active_hiring/total*100:.1f}%)")

Total companies                5,690
Currently hiring               1,432 (25.2%)
Active & hiring                1,423 (25.0%)


---
## 8. Team Size Stats

In [10]:
ts = df["team_size"].describe()
print(f"{'Min':<20} {int(ts['min'])}")
print(f"{'25th percentile':<20} {int(ts['25%'])}")
print(f"{'Median':<20} {int(ts['50%'])}")
print(f"{'Mean':<20} {int(ts['mean'])}")
print(f"{'75th percentile':<20} {int(ts['75%'])}")
print(f"{'Max':<20} {int(ts['max'])}")
print(f"{'Missing':<20} {df['team_size'].isnull().sum()}")

Min                  0
25th percentile      2
Median               6
Mean                 48
75th percentile      20
Max                  8600
Missing              100


---
## 9. Geography

In [11]:
regions = df["regions"].str.split("; ").explode().str.strip()
regions = regions[regions != ""]

print("=== Top Regions ===")
for region, count in regions.value_counts().head(15).items():
    print(f"  {region:<40} {count}")

print()

cities = df["location"].str.split(",").str[0].str.strip()
print("=== Top Cities ===")
for city, count in cities.value_counts().head(15).items():
    print(f"  {city:<40} {count}")

=== Top Regions ===
  America / Canada                         4056
  United States of America                 3920
  Remote                                   2974
  Partly Remote                            1819
  Fully Remote                             1155
  Unspecified                              418
  Europe                                   386
  South Asia                               218
  Latin America                            217
  India                                    206
  United Kingdom                           161
  Canada                                   145
  Southeast Asia                           105
  Mexico                                   90
  Africa                                   87

=== Top Cities ===
  San Francisco                            2111
  New York                                 549
                                           512
  London                                   133
  Los Angeles                              110
  Bengaluru     

---
## 10. Tag Coverage

In [12]:
all_tags = df["tags"].str.split("; ").explode().str.strip()
all_tags = all_tags[(all_tags != "") & all_tags.notnull()]

no_tags = (df["tags"].isna() | (df["tags"] == "")).sum()

print(f"{'Unique tags':<25} {all_tags.nunique()}")
print(f"{'Companies with tags':<25} {len(df) - no_tags} ({(len(df) - no_tags)/len(df)*100:.1f}%)")
print(f"{'Companies without tags':<25} {no_tags} ({no_tags/len(df)*100:.1f}%)")
print(f"{'Avg tags per company':<25} {all_tags.groupby(level=0).count().mean():.1f}")

Unique tags               333
Companies with tags       5094 (89.5%)
Companies without tags    596 (10.5%)
Avg tags per company      3.0


---
## 11. Sample Records

In [13]:
print("=== 5 Random Active Companies ===")
df[df["status"] == "Active"].sample(5, random_state=42)[["name", "one_liner", "batch", "team_size", "industry", "location"]]

=== 5 Random Active Companies ===


,name,one_liner,batch,team_size,industry,location
3395,IMT Care,Disrupting Indian Insurance space by empowerin...,Winter 2022,23.0,Healthcare,"MH, India"
2969,Repool,Modernizing hedge fund back office software an...,Summer 2021,11.0,Fintech,"New York, NY, USA"
3676,Signatur Biosciences,Simple tests for complex diseases.,Summer 2022,5.0,Healthcare,"London, England, United Kingdom"
2548,Laudspeaker,Open source mobile marketing platform. Alterna...,Winter 2021,3.0,B2B,"Mountain View, CA, USA; Remote"
5510,Zalos,Computer Agents for Finance tasks like reconci...,Fall 2025,2.0,B2B,


In [14]:
print("=== 5 Random Hiring Companies ===")
df[df["is_hiring"] == 1].sample(5, random_state=42)[["name", "one_liner", "website", "batch", "team_size", "location"]]

=== 5 Random Hiring Companies ===


,name,one_liner,website,batch,team_size,location
1213,AlemHealth,AI Radiologist in a Box - We provide a vertica...,https://www.alemhealth.com,Winter 2017,25.0,"Singapore, Singapore"
4933,Canvas,Claude Code for GTM Engineering,https://www.canvas.inc/,Fall 2024,3.0,
2725,Infracost,Shift FinOps Left: Proactively Find & Fix Clou...,https://infracost.io,Winter 2021,20.0,Remote
2347,Arist,Deliver effective learning at scale,https://www.arist.co,Summer 2020,23.0,"New York, NY, USA; Remote"
4596,Indemni,Cargo Theft and Fraud Prevention Platform,http://www.indemni.com,Winter 2024,7.0,"San Francisco, CA, USA"
